In [1]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from PIL import Image
import requests

/home/lehoangviet/miniconda3/envs/virenv1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load model and processor
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct-AWQ",
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)
model.tie_weights()

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct-AWQ", use_fast=True)
tokenizer = processor.tokenizer  # <--- use tokenizer directly for text-only

# Your constant prompt
prompt = "Describe this image in detail."

# Tokenize prompt (correctly)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Get prompt embeddings
with torch.no_grad():
    prompt_embeds = model.get_input_embeddings()(inputs["input_ids"])

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration

# Load the model with float16 for better efficiency
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype=torch.float16, device_map="auto"
)

# Define dummy input parameters
batch_size = 1
seq_length = 10  # Short sequence for text tokens
hidden_size = model.config.hidden_size  # Typically 2048 for Qwen2.5-VL-3B
image_size = 448  # Common image size for Qwen2.5-VL
num_channels = 3  # RGB channels
image_tokens = 256  # Number of visual tokens per image (within 4-16384 range)

# Dummy input_ids (batch_size, seq_length)
# Represents token IDs for a short text sequence
input_ids = torch.randint(
    low=0, high=model.config.vocab_size, size=(batch_size, seq_length), dtype=torch.long
).to("cuda")

# Dummy inputs_embeds (batch_size, seq_length, hidden_size)
# Represents embeddings for the text sequence
inputs_embeds = torch.randn(batch_size, seq_length, hidden_size, dtype=torch.float16).to("cuda")

# Dummy pixel_values (batch_size, num_channels, height, width)
# Represents a single image
pixel_values = torch.randn(
    batch_size, num_channels, image_size, image_size, dtype=torch.float16
).to("cuda")

# Dummy image_grid_thw (batch_size, 3)
# Represents the token grid for the image (tokens, height, width)
image_grid_thw = torch.tensor(
    [[image_tokens, 16, 16]], dtype=torch.long  # [tokens, height, width]
).to("cuda")

# Dummy past_key_values
# Set to None for initial forward pass (no cached states)
past_key_values = None

# Test the forward pass
try:
    outputs = model(
        input_ids=input_ids,
        inputs_embeds=inputs_embeds,
        pixel_values=pixel_values,
        image_grid_thw=image_grid_thw,
        past_key_values=past_key_values,
        use_cache=True,
        return_dict=True,
    )
    print("Forward pass successful!")
    print("Output logits shape:", outputs.logits.shape)
except Exception as e:
    print("Error during forward pass:", str(e))